## Simulating traffic around a lock
In this notebook, we simulate a lock on a network which randomly generated vessels have to pass. We add a pre-coded complex lock object on the graph. Vessels are locked together if they can fit inside the lock, and arrive within the clustering time window.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [3]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph

In [4]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-350600,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(350600,0)))

# add edges
graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-350600, 0),Point(-5000, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-350600, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(350600, 0)])), weight=1, length_m=350600-5000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(350600, 0),Point(5000, 0)])), weight=1, length_m=350600-5000)

# add graph to environment
env.graph = graph

In [5]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure

In [6]:
lock_chamber = IsLockChamber(env=env,
                             lock_depth = 10,
                             name='Lock',
                             gate_open = '0',
                             edge = ('0','1'),
                             geometry = Polygon([Point(-200, -25),Point(-200, 25),Point(200, 25),Point(200, -25)]))

In [7]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   distance_from_edge_start = 0)

In [8]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents

In [9]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
    {}
)

In [10]:
def generate_vessel(
    env,
    name,
    start_node,
    end_node,
    arrival_time,
    vessel_speed=4,
    vessel_length=100,
    vessel_beam=20,
    vessel_draft=5,
    vessel_type="tanker"):
    """
    Creates and returns a Vessel object with a computed route through the environment graph.

    Parameters:
    ----------
    env : Environment
        The simulation environment containing the graph and other context.
    name : str
        Human readabile identifier for the vessel.
    start_node : str or int
        The starting node in the graph (converted to string).
    end_node : str or int
        The destination node in the graph (converted to string).
    arrival_time : pd.Timestamp
        The scheduled arrival time of the vessel at the start node.
    vessel_speed : float, optional
        Speed of the vessel in knots or simulation units (default is 4).
    vessel_length : float, optional
        Length of the vessel in meters (default is 100).
    vessel_beam : float, optional
        Beam (width) of the vessel in meters (default is 20).
    vessel_draft : float, optional
        Draught (depth below waterline) of the vessel in meters (default is 10).
    vessel_type : str, optional
        Type of vessel (e.g., "tanker", "cargo", "container") (default is "tanker").

    Returns:
    -------
    Vessel or None
        A Vessel object initialized with the given parameters and route.
        Returns None if no valid path exists between start_node and end_node.
    """
    
    # Ensure nodes are strings
    start_node = str(start_node)
    end_node = str(end_node)

    try:
        route = nx.dijkstra_path(env.graph, start_node, end_node)
    except nx.NetworkXNoPath:
        print(f"⚠️ No path from {start_node} to {end_node}. Vessel {name} not created.")
        return None

    geometry = env.graph.nodes[start_node]['geometry']

    data_vessel = {
        "env": env,
        "name": name,
        "geometry": geometry,
        "route": route,
        "v": vessel_speed,
        "L": vessel_length,
        "B": vessel_beam,
        "T": vessel_draft,
        "type": vessel_type,
        "arrival_time": arrival_time,
    }

    vessel = Vessel(**data_vessel)

    return vessel

In [11]:
def generate_vessels_with_distributions(
    env,
    num_vessels,
    start_time,
    mean_arrival_rate_up=30.,
    mean_arrival_rate_down=30.,
    seed_up=None,
    seed_down=None):
    """
    Generates a list of vessels with interarrival times drawn from specified distributions
    for upward and downward directions. Supports independent seeding for reproducibility.

    Parameters
    ----------
    env : Environment
        The simulation environment containing the graph and vessel context.
    num_vessels : int
        Total number of vessels to generate. Vessels alternate between up and down directions.
    start_time : pd.Timestamp
        The initial timestamp from which vessel arrivals begin.
    arrival_dist_up : callable, optional
        A function returning interarrival times (in minutes) for upward-moving vessels.
        If None, defaults to an exponential distribution with mean 20 minutes.
    arrival_dist_down : callable, optional
        A function returning interarrival times (in minutes) for downward-moving vessels.
        If None, defaults to an exponential distribution with mean 20 minutes.
    seed_up : int or None, optional
        Seed for the random number generator used in upward direction.
    seed_down : int or None, optional
        Seed for the random number generator used in downward direction.

    Returns
    -------
    list of Vessel
        A list of Vessel objects with assigned routes and arrival times.
        Vessels for which no valid path exists are skipped.
    """

    vessels = []

    # Create independent random generators
    rng_up = np.random.default_rng(seed_up)
    rng_down = np.random.default_rng(seed_down)
    arrival_dist_up = lambda: rng_up.exponential(scale=mean_arrival_rate_up)
    arrival_dist_down = lambda: rng_down.exponential(scale=mean_arrival_rate_down)

    up_time = start_time
    down_time = start_time

    for i in range(num_vessels):
        if i % 2 == 0:
            # Upward direction: -1 → +1
            start_node, end_node = "-1", "+1"
            delta_minutes = arrival_dist_up()
            arrival_time = up_time + pd.Timedelta(minutes=delta_minutes)
            up_time = arrival_time
        else:
            # Downward direction: +1 → -1
            start_node, end_node = "+1", "-1"
            delta_minutes = arrival_dist_down()
            arrival_time = down_time + pd.Timedelta(minutes=delta_minutes)
            down_time = arrival_time

        vessel = generate_vessel(
            env=env,
            name=f"Vessel {i + 1}",
            start_node=start_node,
            end_node=end_node,
            arrival_time=arrival_time
        )

        if vessel:
            vessels.append(vessel)

    return vessels

In [12]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [13]:
vessels = generate_vessels_with_distributions(
    env=env,
    num_vessels=11,
    start_time=simulation_start,
    mean_arrival_rate_up=30., #minutes
    mean_arrival_rate_down=30., #minutes
    seed_up=123,
    seed_down=456
)

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation

In [14]:
env.run()

#### 4. Inspect output

In [15]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber.logbook)

print("'{}' logbook data:".format(lock_chamber.name))  
print('')

display(lock_df)

'Lock' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-02 00:14:44.354414,{},0
1,Lock gate closing stop,2025-01-02 00:19:44.354414,{},0
2,Lock chamber converting start,2025-01-02 00:19:44.354414,{},0
3,Lock chamber converting stop,2025-01-02 00:29:44.354414,{},1
4,Lock gate opening start,2025-01-02 00:29:44.354414,{},1
5,Lock gate opening stop,2025-01-02 00:34:44.354414,{},1
6,Waiting for other vessels in lock start,2025-01-02 00:50:24.527200,{},1
7,Waiting for other vessels in lock stop,2025-01-02 00:56:52.853813,{},1
8,Lock gate closing start,2025-01-02 00:56:52.853813,{},1
9,Lock gate closing stop,2025-01-02 01:01:52.853813,{},1


#### Gantt chart of event table

In [16]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber])
fig = generate_vessel_gantt_chart(df_eventtable)

#### Time-distance diagram of vessels passing the lock and planning info

In [17]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -6050, 
                        xlimmax = 6050,
                        ylimmin = pd.Timestamp('2025-01-01 22:00:00'),
                        ylimmax = pd.Timestamp('2025-01-02 09:00:00'),
                        method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

#### Vessel delays: individual delays and overall average

In [18]:
delays = []
for vessel in vessels:
    vessel_df = pd.DataFrame(vessel.logbook)
    waiting_start = vessel_df[vessel_df.Message.str.match(r"^Waiting .* start$", na=False)]
    waiting_stop = vessel_df[vessel_df.Message.str.match(r"^Waiting .* stop$", na=False)]
    if not waiting_stop.empty:
        delay = (waiting_stop.Timestamp.values-waiting_start.Timestamp.values).sum()
    else:
        delay = pd.Timedelta(seconds=0)
    delays.append(delay)

In [19]:
print(f"The average vessel delay is {np.round(np.mean(delays)/np.timedelta64(1,'s')/60,1)} minutes")

The average vessel delay is 42.0 minutes


#### Simulated intensity (vessels per hour)
'get_vessels_during_leveling':
- Identify locking cycles (by looking at the lock logbook)
- From the vessel list identify which vessels were in the lock during that locking cycle

'calculate_cycle_looptimes':
- Using the info derived from 'get_vessels_during_leveling' calculate looptimes

'calculate_detailed_cycle_time':
- using the info from 'get_vessels_during_leveling' and 'calculate_cycle_looptimes' calculate detailed cycle times

In [20]:
leveling_cycles = opentnsim.lock.logutils.get_vessels_during_leveling(lock_chamber, vessels)
looptimes_df = opentnsim.lock.logutils.calculate_cycle_looptimes(leveling_cycles, vessels)
Tc_df = opentnsim.lock.logutils.calculate_detailed_cycle_time(lock_chamber, vessels, leveling_cycles)

display(pd.DataFrame(leveling_cycles))
display(looptimes_df)
display(Tc_df)

,leveling_start,leveling_stop,vessels_present
0,2025-01-02 00:19:44.354414,2025-01-02 00:29:44.354414,[]
1,2025-01-02 01:01:52.853813,2025-01-02 01:11:52.853813,"[Vessel 2, Vessel 4, Vessel 6]"
2,2025-01-02 01:42:40.046038,2025-01-02 01:52:40.046038,"[Vessel 1, Vessel 3, Vessel 5, Vessel 7]"
3,2025-01-02 02:22:18.814936,2025-01-02 02:32:18.814936,[Vessel 8]
4,2025-01-02 02:54:20.391612,2025-01-02 03:04:20.391612,"[Vessel 9, Vessel 11]"
5,2025-01-02 03:27:59.160511,2025-01-02 03:37:59.160511,[Vessel 10]


,cycle,looptime_seconds
0,1,0.000000
1,2,NaN
2,3,250.000000
3,4,249.999999
4,5,250.000000
5,6,250.000000


,t_l_up,sum_t_i_up,T_close_up,T_waterlevel_up,T_open_up,sum_t_u_up,t_l_down,sum_t_i_down,T_close_down,T_waterlevel_down,T_open_down,sum_t_u_down,Tc_seconds,up_vessels,down_vessels,I_s
0,0.0,0.000000,300.0,600.0,300.0,0.000000,0.000000,728.499399,300.0,600.0,300.0,408.596112,3537.095511,[],"[Vessel 2, Vessel 4, Vessel 6]",3.053353
1,250.0,588.596112,300.0,600.0,300.0,588.596113,249.999999,340.172786,300.0,600.0,300.0,48.596113,4465.961123,"[Vessel 1, Vessel 3, Vessel 5, Vessel 7]",[Vessel 8],4.030487
2,250.0,422.980562,300.0,600.0,300.0,228.596111,250.000000,340.172787,300.0,600.0,300.0,48.596112,3940.345572,"[Vessel 9, Vessel 11]",[Vessel 10],2.740876


In [21]:
for index, row in Tc_df.iterrows():
    print('Locking cycle {} has an intensity of {:.2f} vessels per hour'.format(index+1, row['I_s']))

Locking cycle 1 has an intensity of 3.05 vessels per hour
Locking cycle 2 has an intensity of 4.03 vessels per hour
Locking cycle 3 has an intensity of 2.74 vessels per hour


#### Estimated capacity (vessels per hour)

In [22]:
n_max = 4
vessel_speed_outside_of_lock = 4

# Part III, Ch3, Eq. 3.2 (NB: de helft van de looptime wordt hier effectief geimplementeerd door de sailing to lock te berekenen)
t_sailing_to_lock = lock_chamber.sailing_distance_to_crossing_point/vessel_speed_outside_of_lock
T_entering = t_sailing_to_lock + (n_max-1)*lock_chamber.sailing_in_time_gap_through_gate.total_seconds() + 50/2

# Part III, Ch3, Eq. 3.3
T_operation = lock_chamber.gate_closing_time + lock_chamber.levelling_time + lock_chamber.gate_opening_time

# Part III, Ch3, Eq. 3.4 (NB: de helft van de looptime wordt hier effectief geimplementeerd door de sailing out of lock te berekenen)
t_sailing_out_of_lock = lock_chamber.sailing_distance_to_crossing_point/vessel_speed_outside_of_lock
T_exiting = t_sailing_out_of_lock + (n_max-1)*lock_chamber.sailing_out_time_gap_through_gate.total_seconds() + 350/2

# Part III, Ch3, Eq. 3.1
T_locking = T_entering + T_operation + T_exiting
T_c = 2 * T_locking

C_s = 2*n_max / (T_c/3600)

print(f"The capacity of the lock is {np.round(C_s,1)} vessels per hour")

The capacity of the lock is 5.3 vessels per hour
